# **SPECTRA training notebook**

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import argparse
import yaml
import pickle
import torch
import networkx as nx
import scanpy as sc
import numpy as np
import pandas as pd
import warnings
import os

# Suppress annoying warnings for a clean console
warnings.filterwarnings("ignore")

In [4]:
# torch device setting
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(torch.version.cuda)
    print(torch.cuda.get_device_name())
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

12.1
Tesla V100-SXM2-32GB
Using device: cuda


In [5]:
from spectra.utils import set_seed
from spectra.data import data_preprocessing, build_model_dataloaders_perts_split, compute_weights
from spectra.model import SPECTRA
from spectra.training import train

### **Load config file**

In [6]:
config_path = '../configs/vcc_config.yaml'

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)
print(yaml.dump(config, indent=2, sort_keys=False, default_flow_style=False))

project:
  name: SPECTRA_vcc_new_loss
  seed: 42
  deterministic: false
  use_wandb: false
data:
  adata_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/adata_vcc.h5ad
  gene_list_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/gene_list-vcc.txt
  network_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/FUNGI_new_vcc.tsv
  scgpt_embeddings_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/scGPT_embeddings_all_genes.pkl
  gene_weights_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/gene_weights-vcc.pkl
model:
  architecture: FUNGI_FAGCN_3layers_fixed_VCC
  conv_type: FAGCN
  residual_weight: 0.6
  n_channels: 16
  dropout_p: 0.1
  num_node_features: 1
training:
  batch_size: 48
  lr: 0.001
  n_epochs: 25
  alpha: 3.0
  beta: 0.001
  gamma: 1.0
  eta: 1.0
  test_ratio: 0.2
  val_ratio: 0.1



In [7]:
# Initialization & Seeding - set it to deterministic for reproducibility
set_seed(seed=config['project']['seed'], deterministic=config['project']['deterministic'])

### **Load files**

In [8]:
# Load adata
adata = sc.read_h5ad(config['data']['adata_path'])
#adata.raw = adata.copy()
adata = data_preprocessing(adata,
    logtransform=True, 
    min_cells_per_pert=100)
adata

View of AnnData object with n_obs × n_vars = 220998 × 18020
    obs: 'target_gene', 'guide_id', 'batch', 'n_genes'
    var: 'gene_id', 'n_cells'
    uns: 'log1p'

In [9]:
# scGPT gene embeddings
with open(config['data']['scgpt_embeddings_path'], "rb") as f:
    scgpt_dict = pickle.load(f)

In [10]:
# load GRN
network_data = pd.read_csv(config['data']['network_path'], sep='\t')
G = nx.DiGraph()
for _, edge in network_data.iterrows():
    G.add_edge(edge['source'], edge['target'], weight=edge['weight'])

G.remove_nodes_from([n for n in G.nodes if n not in scgpt_dict])
grn_genes = set(G.nodes)

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)

Number of nodes: 4914
Number of edges: 169075


In [11]:
# Filter adata to match final network genes
gene_list = grn_genes & set(adata.var_names)
if grn_genes != gene_list:
    print('WARNING: some genes in the provided gene list are not included in the grn, or the gene embeddings are missing; filtering them out...')
adata = adata[:, adata.var_names.isin(gene_list)]

In [12]:
# Filter out perturbations that aren't in the gene list
perturbations = list(adata.obs['target_gene'].unique())
perturbations.remove('non-targeting')
perts_not_included = list({
    pert for pert in perturbations
    if any(single_pert not in gene_list for single_pert in pert.split('+'))
})
if len(perts_not_included)>0:
    print(print('WARNING: some perturbed genes are not included in the GRN. Filtering these perturbation samples out...'))
adata = adata[~adata.obs['target_gene'].isin(perts_not_included)].copy()

None


### **Edge indices, embeddings matrix**
Other preparation steps fro SPECTRA

In [13]:
# shuffle data, sanity check
adata = adata[np.random.permutation(adata.n_obs), :]
assert set(G.nodes) == set(adata.var_names), "Nodes in G and adata.var_names differ!"

In [14]:
# Map Edge Index
gene_to_idx = {node: i for i, node in enumerate(adata.var_names)}
edges = [(gene_to_idx[u], gene_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [15]:
# Build embedding matrix
scgpt_dict = {gene_to_idx[k]: v for k, v in scgpt_dict.items() if k in gene_to_idx}
scgpt_dim = len(next(iter(scgpt_dict.values())))
embedding_matrix = torch.zeros((num_nodes, scgpt_dim))
for gene_id, emb in scgpt_dict.items():
    embedding_matrix[gene_id] = torch.tensor(emb, dtype=torch.float32)

### **Dataloaders creation**

In [16]:
perturbations = list(adata.obs['target_gene'].unique())
perturbations.remove('non-targeting')

# Merge config dictionaries for the model/dataloaders
model_config = {**config['model'], **config['training']}
model_config['dataset_size'] = adata.shape[0]
model_config['pert_to_idx'] = {pert: i for i, pert in enumerate(perturbations)}

In [17]:
from spectra.data import build_model_dataloaders_from_perts_list

train_loader, val_loader, test_loader, _, _, _, train_adata, _, test_adata = build_model_dataloaders_from_perts_list(adata, model_config, '../data/VCC_h1_hESC_split_indices.json')

Total Unique Perturbations: 148
Train set: 96 perts | 123256 cells
Val set: 16 perts | 8030 cells
Test set: 36 perts | 9655 cells


In [16]:
train_loader, val_loader, _, _, _, _, train_adata, _, _ = build_model_dataloaders_perts_split(adata, model_config)

Total Unique Perturbations: 131
Train set: 92 perts | 121883 cells
Val set: 13 perts | 11746 cells
Test set: 26 perts | 30421 cells


In [18]:
# Load WMSE Weights
weights_path = config['data']['gene_weights_path']
if os.path.exists(weights_path):
    print('found already existing gene weights dictionary! Loading...')
    with open(weights_path, 'rb') as f:
        gene_weights = pickle.load(f)
else:
    print('No gene weights dictionary found. Calculating...')
    gene_weights = compute_weights(adata, gene_to_idx, cells_per_pert=256)
    with open(weights_path, 'wb') as f:
        pickle.dump(gene_weights, f)

No gene weights dictionary found. Calculating...


Calculating WMSE Weights: 100%|██████████| 112/112 [00:00<00:00, 1282.12it/s]


### **Model initialization**

In [19]:
# Initialize W&B
wandb_support = config['project']['use_wandb']
if wandb_support:
    import wandb
    wandb.login()
    wandb.init(project=config['project']['name'], config=model_config)

In [20]:
model = SPECTRA(
    edge_index=edge_index, 
    num_nodes=num_nodes, 
    device=device, 
    config=model_config,
    gene_embeddings=embedding_matrix,
    gene_weights=gene_weights
).to(device)
print(model)

number of trainable parameters: 17713
SPECTRA(
  (gene_embeddings): Embedding(4914, 512)
  (project_gene): Linear(in_features=512, out_features=16, bias=True)
  (film_layer): GeneExpressionFiLM(
    (scale): Linear(in_features=1, out_features=16, bias=True)
    (shift): Linear(in_features=1, out_features=16, bias=True)
  )
  (add_norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (ko_mlp): MLP(
    (network): Sequential(
      (0): Dropout(p=0.0, inplace=False)
      (1): Linear(in_features=512, out_features=16, bias=True)
      (2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (encoder): VariationalGraphEncoder(
    (conv1): DirFAGCNConv(16)
    (conv2): DirFAGCNConv(16)
    (conv_mu): DirFAGCNConv(16)
    (conv_logstd): DirFAGCNConv(16)
    (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (ln2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (proj_mu): Linear(in_features=16, out_features=16, bias=True)

### **Training**

In [21]:
idx_to_gene = {v: k for k, v in gene_to_idx.items()}
train(
    model=model, 
    train_loader=train_loader, 
    test_loader=val_loader,
    lr=model_config['lr'], 
    n_epochs=model_config['n_epochs'],  
    device=device,
    wandb_support=wandb_support,
    var_names=adata.var_names.tolist(),
    idx_to_gene=idx_to_gene,
    alpha_weight=model_config['alpha'],
    beta_weight=model_config['beta'],
    gamma_weight=model_config['gamma'],
    eta_weight=model_config['eta']
)

training at epoch 1:   5%|▍         | 119/2527 [01:47<36:20,  1.10it/s]


KeyboardInterrupt: 

In [22]:
# exit
if wandb_support:
    wandb.finish()
print('model trained and ready to go! Enjoy!')

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▄▅▇█
train/cosine_loss,█▄▂▂▁▁
train/global_loss,█▃▂▁▁▁
train/kl_divergence,▁▅▆▇▇█
train/mmd_ctrl,▁▁▁▁▁▁
train/mmd_pert,█▁▁▁▁▁
train/mse,█▂▂▁▁▁
val/AUPRC,▁
val/test_MMD,█▅▄▂▂▁
val/test_WMSE,▁▂▇█▂▄
epoch,6


model trained and ready to go! Enjoy!
